# Assessment 1
## I. Connect to Db2 database on Cloud using Python

Connect Python to a persistent SQL database, retrieve records and export the table
for subsequent analysis. The dataset contains 70,000 cardiovascular patient records.

**Environment:** Python and SQLite, the approved database alternative. The optional
IBM Db2 connection is included; its cloud execution has not been performed.


### 1. Import the libraries

`sqlite3` is Python's built-in SQLite library. `pandas` formats the returned records
for display. The accompanying `database_lab.py` contains the reusable connection,
cursor, query, export and verification functions.

In [1]:
from pathlib import Path
import json
import sqlite3
import pandas as pd
from IPython.display import display, Markdown
from database_lab import ROOT, RAW_CSV, COLUMNS, source_records, query_specs, run_sqlite_lab

print("SQLite library imported. SQLite version:", sqlite3.sqlite_version)
print("Dataset:", RAW_CSV.name)

SQLite library imported. SQLite version: 3.53.1
Dataset: cardio_train_raw.csv


### 2. Inspect the supplied dataset

The file uses semicolons as separators. Age is in **days**, height in centimetres,
and weight in kilograms. `CARDIO` is the binary target. `ID` identifies the record.
Preserve raw values for the database lab; converting age for display must not alter
the source. Numeric gender codes are retained without assigning unverified labels.

In [2]:
records = source_records()
raw = pd.DataFrame(records, columns=COLUMNS)
print(f"Source contains {len(raw):,} records and {len(raw.columns)} columns.")
display(raw.head(10))
display(raw.isna().sum().rename("MISSING_VALUES").to_frame())
print("Unique IDs:", raw["ID"].nunique())
print("Disease cases:", int(raw["CARDIO"].sum()))
print("Age remains in days in the database and CSV export.")

Source contains 70,000 records and 13 columns.


,ID,AGE,GENDER,HEIGHT,WEIGHT,AP_HI,AP_LO,CHOLESTEROL,GLUC,SMOKE,ALCO,ACTIVE,CARDIO
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0
5,8,21914,1,151,67.0,120,80,2,2,0,0,0,0
6,9,22113,1,157,93.0,130,80,3,1,0,0,1,0
7,12,22584,2,178,95.0,130,90,3,3,0,0,1,1
8,13,17668,1,158,71.0,110,70,1,1,0,0,1,0
9,14,19834,1,164,68.0,110,60,1,1,0,0,0,0


,MISSING_VALUES
ID,0
AGE,0
GENDER,0
HEIGHT,0
WEIGHT,0
AP_HI,0
AP_LO,0
CHOLESTEROL,0
GLUC,0
SMOKE,0


Unique IDs: 70000
Disease cases: 34979
Age remains in days in the database and CSV export.


### 3. Connect, create the table and retrieve data using a cursor

The database is stored at `data/cardio_train.sqlite`. On the first run, the supplied
records are inserted into `CARDIO_TRAIN`. On later runs, the existing records are
compared with the source CSV and reused. `ID` is the primary key.

The workflow uses these Python operations:

```python
connection = sqlite3.connect(database_path)
cursor = connection.cursor()
try:
    cursor.execute("SELECT COUNT(*) AS TOTAL_ROWS FROM CARDIO_TRAIN")
    rows = cursor.fetchall()
finally:
    cursor.close()
    connection.close()
```

The actual function below performs all queries, saves each result, exports the complete
table and closes the connection in a `finally` block. It then reopens the database
read-only and verifies integrity and persistence.

In [3]:
results, status = run_sqlite_lab()
print("Connected to the local SQLite database and completed the SQL workflow.")
print("Database:", "data/cardio_train.sqlite")
print("Connection closed:", status["connection_closed"])
print("Reopened database integrity:", status["integrity_check"])
print("Every source record matches the saved database:", status["all_source_records_match"])
database_preview = next(item for item in results if item["id"] == "Q11")
display(pd.DataFrame(database_preview["rows"], columns=database_preview["columns"]))


Connected to the local SQLite database and completed the SQL workflow.
Database: data/cardio_train.sqlite
Connection closed: True
Reopened database integrity: ok
Every source record matches the saved database: True


,ID,AGE,GENDER,HEIGHT,WEIGHT,AP_HI,AP_LO,CHOLESTEROL,GLUC,SMOKE,ALCO,ACTIVE,CARDIO
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0
5,8,21914,1,151,67.0,120,80,2,2,0,0,0,0
6,9,22113,1,157,93.0,130,80,3,1,0,0,1,0
7,12,22584,2,178,95.0,130,90,3,3,0,0,1,1
8,13,17668,1,158,71.0,110,70,1,1,0,0,1,0
9,14,19834,1,164,68.0,110,60,1,1,0,0,0,0


### 4. Export for offline analysis and verify the closed connection

The export is written from the SQL result set rather than copied from the input CSV.
It has comma separators, the original 13 columns, age in days and no extra index
column. Every exported value is checked against the source. The saved SQLite file
can be opened again after the original connection has been closed.

In [4]:
export = pd.read_csv(ROOT / "data" / "CARDIO_TRAIN_export.csv")
print("SQL export:", "data/CARDIO_TRAIN_export.csv")
print("Exported records:", len(export))
print("Export matches every source record:", status["export_matches_source"])
print("Original connection closed:", status["connection_closed"])
print("Verification connection closed:", status["reopened_connection_closed"])
display(pd.DataFrame([status]))

SQL export: data/CARDIO_TRAIN_export.csv
Exported records: 70000
Export matches every source record: True
Original connection closed: True
Verification connection closed: True


,engine,cloud_executed,lecturer_approved_local_alternative,source_sha256,executed_at_utc,all_source_records_match,row_count,disease_cases,integrity_check,connection_closed,reopened_connection_closed,export_matches_source,executed_queries,export_rows,success
0,SQLite,False,True,21a705d23381b0dfd6a6416da701b490744f1fc3b47e9f...,2026-11-05T19:35:47.907008+00:00,True,70000,34979,ok,True,True,True,13,70000,True


### 5. Findings

The table contains 34,979 disease cases and 35,021 records without disease.
Gender-code counts are 45,530 for code 1 and 24,470 for code 2.
Raw blood-pressure outliers are preserved in the database; Assessment 3 examines
their treatment. Age remains in days in the stored data.


### 6. Optional IBM Db2 Cloud connection

Use this section with an available Db2 instance and private service credentials.

The helper imports `ibm_db` and `ibm_db_dbi`, builds an SSL DSN, calls
`ibm_db.connect(dsn, "", "")`, retrieves server and client metadata using
`ibm_db.server_info` and `ibm_db.client_info`, retrieves the table through a DB-API
cursor, exports it, and calls `ibm_db.close` in a `finally` block.

For actual Cloud execution, first import the raw CSV into `CARDIO_TRAIN` in your
Db2 instance and install `requirements-db2.txt`. Set the flags below to enable
the connection and the private credential prompts. Environment variables can
be used instead. No credentials from the assignment PDF are used.

```python
dsn = (
    "DRIVER={IBM DB2 ODBC DRIVER};"
    "DATABASE=...;HOSTNAME=...;PORT=...;PROTOCOL=TCPIP;"
    "UID=...;PWD=...;SECURITY=SSL;"
)
conn = ibm_db.connect(dsn, "", "")
try:
    server = ibm_db.server_info(conn)
    client = ibm_db.client_info(conn)
finally:
    ibm_db.close(conn)
```

Actual implementation: `database_lab.py`, function `run_db2_lab`.
The connection string and password are never printed. Cloud exports and evidence
are stored separately from the local results.

In [5]:
RUN_DB2 = False
ENTER_DB2_CREDENTIALS = False

if RUN_DB2:
    import os
    from getpass import getpass
    import ibm_db
    from database_lab import run_db2_lab

    if ENTER_DB2_CREDENTIALS:
        os.environ["DB2_DATABASE"] = input("Database name (e.g. BLUDB): ").strip()
        os.environ["DB2_HOSTNAME"] = input("IBM Db2 hostname: ").strip()
        os.environ["DB2_PORT"] = input("IBM SSL port: ").strip()
        os.environ["DB2_UID"] = input("Database username: ").strip()
        os.environ["DB2_PWD"] = getpass("Database password: ")
        os.environ["DB2_TABLE"] = input("Table (Enter for CARDIO_TRAIN): ").strip() or "CARDIO_TRAIN"
        certificate = input("Certificate file if required (otherwise Enter): ").strip()
        os.environ["DB2_SSL_CERT"] = certificate

    cloud_results, cloud_status = run_db2_lab()
    for item in cloud_results:
        display(Markdown(f'**{item["id"]}: {item["title"]}**'))
        display(pd.DataFrame(item["rows"], columns=item["columns"]))
    print("Cloud queries and export completed:", cloud_status["success"])
else:
    print("Db2 Cloud execution skipped. Saved assessment results use approved local SQLite.")

Db2 Cloud execution skipped. Saved assessment results use approved local SQLite.


### Results

The database contains 70,000 records and 13 original columns. All values match
the supplied CSV. The complete SQL export is available for Assessments 2 and 3.
Both the original connection and the read-only verification connection are closed.

| Objective | Implementation |
| --- | --- |
| Import a database library | `sqlite3` |
| Establish a connection | Persistent `data/cardio_train.sqlite` |
| Retrieve and display records | Cursor queries and dataset preview |
| Retrieve data for further analysis | Complete, verified SQL CSV export |
| Close the connection | `finally` blocks and read-only reopen check |
| IBM Db2 API | Optional SSL connection, server and driver metadata |

References: supplied ADY201m brief, section I;
[Python sqlite3](https://docs.python.org/3/library/sqlite3.html);
[IBM Db2 driver](https://github.com/ibmdb/python-ibmdb).
